In [ ]:
import numpy as np
import pandas as pd
import torch
import h5py
import joblib
import os, sys
import subprocess
import matplotlib.pyplot as plt
import math, json

In [ ]:
import sys
sys.path.insert(0, "/mnt/efsUCE/umap_pytorch")  #git+https://github.com/ucscBrainExplorer/umap_pytorch.git
import umap_pytorch
print(umap_pytorch.__file__)

In [ ]:
from s3dir import workdir, push

results_dir = "s3://latentbrain/HMBAv0.5_ucebrain768bicans2048_shuffle/"

embedding_h5_file = "pca_corrected_species_k8.h5"
embedding_h5_file_key = "pca"
metadata_file = "obs.tsv.gz"
knn_outfile = "knn_30_species_k8_pca.npz"
k = 10
batch_size = 512 # GPU memory bound; 2048 trained fine on a 24GB A10G

existing_model_iteration = 0
epochs = 1

# One results_dir can hold several tries (e.g. different batch corrections). run_tag goes into every
# output name: pytorch_k3_e5.pkl, transform_k3_e5.json, umap_k3_e5.npy. "" keeps the old names.
run_tag = "k8_pca"
tag = f"_{run_tag}" if run_tag else ""
# Training on a batch-corrected PCA (e.g. pca_corrected_k3.h5): set its directions file (batch_dirs_k3.npy)
# so the transform config gets "d_hat" and map_newdata.py removes the same directions from new data.
d_hat_file = "species_dirs_k8.npy"

# A local folder is used as is; an S3 prefix (s3://latentbrain/...) is synced into a local
# cache, fetching only the files training reads (obs.tsv.gz: the metadata_file of the plot cells)
results_dir = workdir(results_dir, files=[embedding_h5_file, knn_outfile, metadata_file,
                                          f"pytorch{tag}_e{existing_model_iteration}.pkl"])
tag

## Load Embedding back into memory

In [ ]:
# Load the embedding data
# Open the HDF5 file
with h5py.File(os.path.join(results_dir, embedding_h5_file), "r") as f:
    X_data = f[embedding_h5_file_key][:]

# Convert to PyTorch tensor so to use gpu
data_tensor = torch.from_numpy(X_data)
data_tensor, data_tensor.shape

## Load Knn Back

In [ ]:
# load knn back
data = np.load(os.path.join(results_dir, knn_outfile))
knn_indices = data["indices"]
knn_distances = data["distances"]

In [ ]:
knn_distances = np.sqrt(knn_distances)

## PUMAP 
using umap_pytorch https://github.com/ucscBrainExplorer/umap_pytorch.git

## set up training on all cells

In [ ]:
knn_indices = knn_indices[:,:k]
knn_dists = knn_distances[:,:k]

### Load model back or start with a new model

In [ ]:
if existing_model_iteration > 0:
    from umap_pytorch import load_pumap
    path = os.path.join(results_dir, f'pytorch{tag}_e{existing_model_iteration}.pkl')
    pumap = load_pumap(path)
    pumap.epochs = epochs
    print ("Start with existing model", f'pytorch{tag}_e{existing_model_iteration}.pkl')
else:
    import torch.nn.functional as F
    pumap = umap_pytorch.PUMAP(
        encoder=None,           # nn.Module, None for default
        decoder=None,           # nn.Module, True for default, None for encoder only
        n_neighbors=10,
        min_dist= 0.2, # default 0.1
        metric="euclidean",
        n_components=2,
        beta=1.0,               # How much to weigh reconstruction loss for decoder
        reconstruction_loss=F.binary_cross_entropy_with_logits, # pass in custom reconstruction loss functions
        random_state=None,
        lr=1e-3,
        epochs=epochs, #5,
        batch_size= batch_size,
        num_workers=0, #10,
        num_gpus=1,
        match_nonparametric_umap=False, # Train network to match embeddings from non parametric umap,
    )
    print ("Start with a new model")

### Train

In [ ]:
existing_model_iteration, pumap.epochs

In [ ]:
np.product = np.prod  # alias to fix compatibility
pumap.fit(data_tensor, knn_indices, knn_dists)

### Saving model

In [ ]:
existing_model_iteration = existing_model_iteration+ pumap.epochs
existing_model_iteration

In [ ]:
model_path = os.path.join(results_dir, f'pytorch{tag}_e{existing_model_iteration}.pkl')
pumap.save(model_path)

### Quick viz X_umap_pytorch

In [ ]:
# load model
from umap_pytorch import load_pumap
model_path = os.path.join(results_dir, f'pytorch{tag}_e{existing_model_iteration}.pkl')
pumap = load_pumap(model_path)
print("Loaded model", f'pytorch{tag}_e{existing_model_iteration}.pkl')

#### random 500K

In [ ]:
view_size = 500_000
if X_data.shape[0] < view_size:
    subset_indices = range (0, X_data.shape[0])
else:
    np.random.seed(42)  # set seed for reproducibility
    subset_indices = np.random.choice(X_data.shape[0], size= view_size, replace=False)

In [ ]:
X_umap_pytorch = pumap.transform(data_tensor[subset_indices])

In [ ]:
# current
plt.scatter(X_umap_pytorch[:,0], X_umap_pytorch[:,1], alpha =0.5, s=0.1)
plt.show()

## optional quick viz with metadata

In [ ]:
usecols = ["concensus_class", "species", "n_genes", "concensus_subclass"]

if "metadata" not in globals():
    metadata = pd.read_csv(os.path.join(results_dir, metadata_file), sep='\t', usecols=usecols)

In [ ]:
metadata.columns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

metadata_feature = "species"
top_n = 20

colors = metadata[metadata_feature].iloc[subset_indices].astype(str).to_numpy()
top_cats = list(pd.Series(colors).value_counts().nlargest(top_n).index)
labels = np.where(np.isin(colors, top_cats), colors, "other")

palette = plt.get_cmap("tab20").colors
color_map = {cat: palette[i] for i, cat in enumerate(top_cats)}

is_other = labels == "other"
rng = np.random.default_rng(0)
idx = rng.permutation(np.flatnonzero(~is_other))

fig, ax = plt.subplots(figsize=(7, 7))
# "other" in light grey, underneath
ax.scatter(X_umap_pytorch[is_other, 0], X_umap_pytorch[is_other, 1],
           color=(0.85, 0.85, 0.85), s=0.3, alpha=0.3, rasterized=True)
# top categories in shuffled order on top
ax.scatter(X_umap_pytorch[idx, 0], X_umap_pytorch[idx, 1],
           c=[color_map[l] for l in labels[idx]], s=0.3, alpha=0.5, rasterized=True)
ax.set_aspect("equal", adjustable="datalim")
ax.axis("off")
ax.set_title(f"UMAP colored by {metadata_feature}")

patches = [mpatches.Patch(color=color_map[c], label=c) for c in top_cats]
if is_other.any():
    patches.append(mpatches.Patch(color=(0.85, 0.85, 0.85), label="other"))
fig.legend(handles=patches, title=metadata_feature, loc="lower center",
           bbox_to_anchor=(0.5, 0), ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0.2, 1, 1))  # room for the legend
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

features = usecols
top_n = 10
rng = np.random.default_rng(0)

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()

for ax, metadata_feature in zip(axes, features):
    values = metadata[metadata_feature].iloc[subset_indices]

    # ---- continuous features (e.g. n_genes) ----
    if pd.api.types.is_numeric_dtype(values) and not pd.api.types.is_bool_dtype(values):
        vals = values.to_numpy(dtype=float)
        ok = np.isfinite(vals)

        # clip to 1st-99th percentile so outliers don't wash out the colors
        vmin, vmax = np.nanpercentile(vals[ok], [1, 99])

        # NaNs in grey underneath
        ax.scatter(X_umap_pytorch[~ok, 0], X_umap_pytorch[~ok, 1],
                   color=(0.85, 0.85, 0.85), alpha=0.3, s=0.3, rasterized=True)

        idx = rng.permutation(np.flatnonzero(ok))
        sc = ax.scatter(X_umap_pytorch[idx, 0], X_umap_pytorch[idx, 1],
                        c=vals[idx], cmap="viridis", vmin=vmin, vmax=vmax,
                        alpha=0.5, s=0.3, rasterized=True)
        cbar = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
        cbar.set_label(metadata_feature)
        cbar.solids.set_alpha(1)   # colorbar not faded by the point alpha
        ax.set_title(metadata_feature)

    # ---- categorical features ----
    else:
        colors = values.astype(str).to_numpy()
        top_cats = pd.Series(colors).value_counts().nlargest(top_n).index
        labels = np.where(np.isin(colors, top_cats), colors, "other")
        unique_cats = list(top_cats)

        cmap = plt.get_cmap("tab10")
        color_map = {cat: cmap(i) for i, cat in enumerate(unique_cats)}
        is_other = labels == "other"

        ax.scatter(X_umap_pytorch[is_other, 0], X_umap_pytorch[is_other, 1],
                   color=(0.85, 0.85, 0.85), alpha=0.3, s=0.3, rasterized=True)

        idx = rng.permutation(np.flatnonzero(~is_other))
        ax.scatter(X_umap_pytorch[idx, 0], X_umap_pytorch[idx, 1],
                   c=[color_map[l] for l in labels[idx]], alpha=0.5, s=0.3, rasterized=True)

        patches = [mpatches.Patch(color=color_map[c], label=c) for c in unique_cats]
        ax.legend(handles=patches, title=f"top {top_n} {metadata_feature}",
                  bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0, fontsize=8)
        ax.set_title(f"top {top_n} {metadata_feature}")

    ax.axis("off")

plt.tight_layout()
plt.show()

## Save UMAP

In [ ]:
n_samples = data_tensor.shape[0]
embedding_dim = 2  # since n_components=2
X_umap_pytorch = np.zeros((n_samples, embedding_dim), dtype=np.float32)

# --- Batch transform ---
print("Starting batch transformation ...", flush=True)
batch_size = 1_000_000
for start in range(0, n_samples, batch_size):
    end = min(start + batch_size, n_samples)
    batch = data_tensor[start:end]
    emb = pumap.transform(batch)
    X_umap_pytorch[start:end] = emb
    print(f"Processed {end}/{n_samples}", flush=True)

In [ ]:
# File names only, no folder: map_newdata looks for them next to this JSON, so the
# JSON works wherever the folder is copied (EFS, the NRP PVC, S3, courtyard)
config = {}
config["pumap_model_file"] = f'pytorch{tag}_e{existing_model_iteration}.pkl'
config["pca_model_file"] = "ipca_model.pkl" # hard-coded for ipca transformation
# Training inputs, for the record
config["pca_file"] = embedding_h5_file
config["knn_file"] = knn_outfile
config["knn_k"] = 30
config["knn_nprobe"] = 20
config["pca_n_components"] =50

# --- Axis adjustment ---
UMAP1_size = np.ptp(X_umap_pytorch[:, 0])
UMAP2_size = np.ptp(X_umap_pytorch[:, 1])
if UMAP1_size < UMAP2_size:
    print("Swapping axes (too tall UMAP)...", flush=True)
    X_umap_pytorch[:, [0, 1]] = X_umap_pytorch[:, [1, 0]]
    config["swap_xy_matrix"] =  [
        [0, 1],
        [1, 0]
    ]
else:
    config["swap_xy_matrix"] =  [
        [1, 0],
        [0, 1]
    ]
config["PUMAP_k"]=k
if d_hat_file:
    config["d_hat"] = d_hat_file
config

In [ ]:
# --- Saveing transformation config ----
config_path = os.path.join(results_dir, f"transform{tag}_e{existing_model_iteration}.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

In [ ]:
# --- Save umap ----
umap_file = f"umap{tag}_e{existing_model_iteration}.npy"
output_path = os.path.join(results_dir, umap_file)
np.save(output_path, X_umap_pytorch)

In [ ]:
# Upload new or changed outputs when results_dir is an S3 prefix (no-op for a local folder)
push(results_dir)